In [1]:
import planetary_computer
import itertools
import dask.dataframe as dd
import pandas as pd

cc = planetary_computer.get_container_client("pcstacitems", "items")

blobs = list(cc.list_blobs("landsat-c2-l2.parquet/"))

# def key(blob):
#     return blob.name.split("/")[1].split("_")[0]

# keep_blobs = []
# for k, v in itertools.groupby(sorted(blobs, key=key), key=key):
#     v = list(v)
#     blob = max(v, key=lambda x: x.last_modified)
#     keep_blobs.append(blob)
    
# uris = [f"az://items/{blob.name}" for blob in keep_blobs]

In [2]:
# Date filters
# LS4: July 1982 - June 2001
# LS5: March 1984 - June 2013
# LS7: April 1999 - present (but with SLC failure in 2003)
# LS8: February 2013 - present
# LS9: September 2021 - present
# LS4 and LS5 have TM sensors: band 1 (blue), band 2 (green), band 3 (red), band 4 (NIR), band 5 (SWIR1), band 7 (SWIR2)
# LS7 has ETM+ sensors: band 1 (blue), band 2 (green), band 3 (red), band 4 (NIR), band 5 (SWIR1), band 7 (SWIR2)
# LS8 and LS9 have OLI sensors: band 1 (coastal), band 2 (blue), band 3 (green), band 4 (red), band 5 (NIR), band 6 (SWIR1), band 7 (SWIR2)

# Likely use LS4 and LS5
max_date = "2013-06-30"
min_date = "1982-07-01"

# Grab blobs that are within the date range
keep_blobs = []
for blob in blobs:
    date = blob.name.split("/")[1].split("_")[1]
    if min_date <= date <= max_date:
        keep_blobs.append(blob)
        print(blob.name)

uris = [f"az://items/{blob.name}" for blob in keep_blobs]

landsat-c2-l2.parquet/part-000_1982-08-22T14:18:20.392044+00:00_1982-12-31T18:15:03.413020+00:00.parquet
landsat-c2-l2.parquet/part-001_1983-01-01T15:37:00.097094+00:00_1983-11-18T18:03:15.631076+00:00.parquet
landsat-c2-l2.parquet/part-002_1984-03-05T15:38:45.602013+00:00_1984-12-31T18:44:02.181007+00:00.parquet
landsat-c2-l2.parquet/part-003_1985-01-01T13:04:30.416019+00:00_1985-12-12T17:42:46.748020+00:00.parquet
landsat-c2-l2.parquet/part-004_1986-01-03T07:12:24.383038+00:00_1986-12-31T17:35:15.892057+00:00.parquet
landsat-c2-l2.parquet/part-005_1987-01-01T06:42:46.346025+00:00_1987-12-31T17:57:52.362064+00:00.parquet
landsat-c2-l2.parquet/part-006_1988-01-01T07:59:51.500094+00:00_1988-12-31T21:15:02.889007+00:00.parquet
landsat-c2-l2.parquet/part-007_1989-01-01T06:35:14.330081+00:00_1989-12-31T19:59:30.094026+00:00.parquet
landsat-c2-l2.parquet/part-008_1990-01-01T07:35:49.247013+00:00_1990-12-31T16:17:27.226014+00:00.parquet
landsat-c2-l2.parquet/part-009_1991-01-01T07:19:47.1660

In [3]:
print(len(uris))

32


In [4]:
# All columns: Index(['assets', 'bbox', 'collection', 'geometry', 'id', 'links',
#    'stac_extensions', 'stac_version', 'type', 'created', 'datetime',
#    'description', 'eo:cloud_cover', 'gsd', 'instruments',
#    'landsat:cloud_cover_land', 'landsat:collection_category',
#    'landsat:collection_number', 'landsat:correction', 'landsat:scene_id',
#    'landsat:wrs_path', 'landsat:wrs_row', 'landsat:wrs_type', 'platform',
#    'proj:epsg', 'proj:shape', 'proj:transform', 'sci:doi',
#    'view:off_nadir', 'view:sun_azimuth', 'view:sun_elevation'],
#   dtype='object')

df = dd.read_parquet(uris, 
                     columns = ["id", "datetime", "geometry", "bbox", "eo:cloud_cover", "platform", "landsat:collection_number", "landsat:scene_id", "landsat:wrs_path", "landsat:wrs_row", "instruments", "proj:epsg", "proj:shape", "proj:transform", "sci:doi", "view:off_nadir", "view:sun_azimuth", "view:sun_elevation"],
                     storage_options={"account_name": "pcstacitems", "credential": planetary_computer.sas.get_token("pcstacitems", "items").token})
df.head()

,id,datetime,geometry,bbox,eo:cloud_cover,platform,landsat:collection_number,landsat:scene_id,landsat:wrs_path,landsat:wrs_row,instruments,proj:epsg,proj:shape,proj:transform,sci:doi,view:off_nadir,view:sun_azimuth,view:sun_elevation
0,LT04_L2SP_037030_19821230_02_T1,1982-12-30 17:29:45.886000+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -110.86815155, 'ymin': 42.16322498, '...",8.0,landsat-4,02,LT40370301982364XXX01,037,030,[tm],32612,"[7081, 7831]","[30.0, 0.0, 510885.0, 0.0, -30.0, 4884615.0]",10.5066/P9IAXOVV,0,153.193181,18.969958
1,LT04_L2SP_037031_19821128_02_T1,1982-11-28 17:30:07.254044+00:00,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...","{'xmin': -111.39839774, 'ymin': 40.78781495, '...",48.0,landsat-4,02,LT40370311982332XXX01,037,031,[tm],32612,"[7101, 7851]","[30.0, 0.0, 467385.0, 0.0, -30.0, 4731015.0]",10.5066/P9IAXOVV,0,155.242596,22.950310
2,LT04_L2SP_037031_19821214_02_T1,1982-12-14 17:30:01.944025+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -111.33233769, 'ymin': 40.76756495, '...",47.0,landsat-4,02,LT40370311982348XXX01,037,031,[tm],32612,"[7101, 7851]","[30.0, 0.0, 472785.0, 0.0, -30.0, 4728915.0]",10.5066/P9IAXOVV,0,154.407765,20.676323
3,LT04_L2SP_037031_19821230_02_T2,1982-12-30 17:30:09.568025+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -111.3140177, 'ymin': 40.76995495, 'x...",1.0,landsat-4,02,LT40370311982364XXX05,037,031,[tm],32612,"[7091, 7841]","[30.0, 0.0, 474285.0, 0.0, -30.0, 4728915.0]",10.5066/P9IAXOVV,0,152.624197,20.101315
4,LT04_L2SP_011031_19821208_02_T2,1982-12-08 14:49:27.753038+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -71.2485277, 'ymin': 40.77333499, 'xm...",5.0,landsat-4,02,LT40110311982342PAC00,011,031,[tm],32619,"[7201, 7951]","[30.0, 0.0, 315885.0, 0.0, -30.0, 4731915.0]",10.5066/P9IAXOVV,0,154.862849,21.352277


In [5]:
df.columns

Index(['id', 'datetime', 'geometry', 'bbox', 'eo:cloud_cover', 'platform',
       'landsat:collection_number', 'landsat:scene_id', 'landsat:wrs_path',
       'landsat:wrs_row', 'instruments', 'proj:epsg', 'proj:shape',
       'proj:transform', 'sci:doi', 'view:off_nadir', 'view:sun_azimuth',
       'view:sun_elevation'],
      dtype='object')

In [6]:
print(len(df))

4669555


In [7]:
# df = df[['id', 'geometry', 'bbox', 'datetime', 'eo:cloud_cover', "s2:product_uri", "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]
df = df[["id", "datetime", "geometry", "bbox", "eo:cloud_cover", "platform", "landsat:collection_number", "landsat:scene_id", "landsat:wrs_path", "landsat:wrs_row", "instruments"]]

df["datetime"] = dd.to_datetime(df["datetime"])

In [8]:
df.head()

,id,datetime,geometry,bbox,eo:cloud_cover,platform,landsat:collection_number,landsat:scene_id,landsat:wrs_path,landsat:wrs_row,instruments
0,LT04_L2SP_037030_19821230_02_T1,1982-12-30 17:29:45.886000+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -110.86815155, 'ymin': 42.16322498, '...",8.0,landsat-4,02,LT40370301982364XXX01,037,030,[tm]
1,LT04_L2SP_037031_19821128_02_T1,1982-11-28 17:30:07.254044+00:00,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...","{'xmin': -111.39839774, 'ymin': 40.78781495, '...",48.0,landsat-4,02,LT40370311982332XXX01,037,031,[tm]
2,LT04_L2SP_037031_19821214_02_T1,1982-12-14 17:30:01.944025+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -111.33233769, 'ymin': 40.76756495, '...",47.0,landsat-4,02,LT40370311982348XXX01,037,031,[tm]
3,LT04_L2SP_037031_19821230_02_T2,1982-12-30 17:30:09.568025+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -111.3140177, 'ymin': 40.76995495, 'x...",1.0,landsat-4,02,LT40370311982364XXX05,037,031,[tm]
4,LT04_L2SP_011031_19821208_02_T2,1982-12-08 14:49:27.753038+00:00,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,"{'xmin': -71.2485277, 'ymin': 40.77333499, 'xm...",5.0,landsat-4,02,LT40110311982342PAC00,011,031,[tm]


In [10]:
filtered_df = df[(df['datetime'] < max_date) &
                 (df['datetime'] >= min_date) &
                 (df["eo:cloud_cover"] < 20) &
                 (df["platform"].isin(["landsat-4", "landsat-5"]))]

In [11]:
filtered_df = filtered_df.compute()
print(len(filtered_df))

1168520


In [14]:
max_date = filtered_df['datetime'].max().strftime("%Y_%m_%d")
print(max_date)

2012_05_05


In [16]:
min_date = filtered_df['datetime'].min().strftime("%Y_%m_%d")
print(min_date)

1982_08_22


In [ ]:
filtered_df.to_parquet(f"landsat_4_5_{min_date}_to_{max_date}.parquet", index=False)
print(f"Saved to: landsat_4_5_{min_date}_to_{max_date}.parquet")

Saved to: landsat_lt_2012_05_05_gt_1982_08_22.parquet
